# 04 - Phase 3: error taxonomy annotation (Weeks 2-3, NEEDS HUMAN INPUT)\n\nStep 8/11. Run the first cell to export `outputs/annotation_sheet_50.csv` (N=50 review cases: disagreement cases + incorrect/low-F).\n\n**Human task:** BOTH annotators independently fill the `primary_label` and optional `secondary_label` columns (categories: knowledge_gap, reasoning_slip, hallucinated_premise, question_misreading, confident_unjustified_leap, other) and save as `labels_annotator1.csv` and `labels_annotator2.csv`. Upload both back into `outputs/` and run the last cell to compute agreement (raw % + Cohen's kappa) and the final adjudicated labels.

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


In [ ]:
import json
from src.common import load_config, read_jsonl, read_records, save_json
from src.taxonomy import export_annotation_sheet, select_review_cases
cfg = load_config()
n = cfg["dataset"]["n_sample"]
data_dir = cfg["paths"]["data_dir"]
out_dir = "outputs"
scored = read_records(f"{out_dir}/scored_{n}.json")
items = read_jsonl(f"{data_dir}/subset_100.jsonl")
traces = read_jsonl(f"{out_dir}/traces_{n}.jsonl")
# full 400-item pool for review cases
all_items = read_jsonl(f"{data_dir}/sample_{n}.jsonl")
off = json.load(open(f"{out_dir}/off_diagonal_ids.json")) if __import__("os").path.exists(f"{out_dir}/off_diagonal_ids.json") else []
case_ids = select_review_cases(scored, off, n_total=50, traces=traces)
save_json(case_ids, f"{out_dir}/review_case_ids.json")
csv_path = export_annotation_sheet(case_ids, all_items, traces, scored,
                                   f"{out_dir}/annotation_sheet_50.csv")
print("Wrote", csv_path, "with", len(case_ids), "cases - download it and annotate.")


In [ ]:
# After uploading labels_annotator1.csv and labels_annotator2.csv into outputs/:
from src.taxonomy import load_labels, compute_agreement, finalize_labels, taxonomy_summary
l1 = load_labels("outputs/labels_annotator1.csv")
l2 = load_labels("outputs/labels_annotator2.csv")
agree = compute_agreement(l1, l2)
print("Inter-annotator agreement:", agree)
# Adjudication: fill this dict with your discussed decision for each disputed id
adjudicated = {cid: "" for cid in agree["disputed_ids"]}  # <-- HUMAN INPUT
final_csv = finalize_labels(l1, l2, adjudicated, "outputs/taxonomy_labels_final.csv")
summary = taxonomy_summary(final_csv, out_json="outputs/taxonomy_summary.json")
print("Taxonomy summary:", summary)


In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)
